In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>검색 증강 생성(RAG)은 문서를 미리 색인하는 흐름과 질문마다 관련 문서를 찾는 흐름으로 나뉩니다. 이 절에서는 검색 문맥이 없을 때 생성 모델이 사내 규정을 답할 수 없는 이유부터 확인합니다.</p>

In [2]:
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>가상 기관의 규정을 검색 문맥 없이 실제 생성 모델에 질문합니다.</p>

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model('openai:gpt-5.4-mini', temperature=0)
question = '가상 기관 파란도서관의 정기 휴관일은 언제인가요?'
reply = model.invoke(question)
print(reply.text)


제가 확인할 수 있는 범위에서는 **“파란도서관”이라는 가상 기관의 정기 휴관일 정보가 제공되어 있지 않습니다.**  
가상 설정이라면 보통 아래처럼 정해집니다:

- **매주 월요일 휴관**
- 또는 **매월 첫째/셋째 주 월요일 휴관**
- 또는 **공휴일 휴관**

원하시면 제가 **“파란도서관”의 운영 규정 예시**를 만들어서 정기 휴관일을 설정해드릴게요.


<h3>변형 실행</h3>
<p><code>question</code>을 다른 가상 기관의 내부 규정 질문으로 바꾸어 한 번 더 실행합니다. 모델이 공개 지식과 사내 문서를 구분해 답하는지 확인합니다.</p>

In [4]:
question = '가상 회사 해솔상사의 경조 휴가는 며칠인가요?'
changed = model.invoke(question)
print(changed.text)


“해솔상사”가 **가상 회사**라면, 제가 가진 정보만으로는 **정해진 경조 휴가 일수**를 알 수 없습니다.

보통은 회사 **취업규칙 / 인사규정 / 복지규정**에 따라 달라집니다.  
예를 들어 경조사 종류별로:

- 본인 결혼
- 배우자 출산
- 부모상
- 조부모상
- 자녀 결혼

등에 따라 **1~5일 이상**으로 다르게 정해지는 경우가 많습니다.

원하시면 제가  
1) **해솔상사용 경조휴가 규정 예시**를 만들어드리거나,  
2) **실제 회사 취업규칙 스타일**로 정리해드릴게요.


<h3>일부러 실패해 보기</h3>
<p>사내 문서를 주지 않았으므로 출처 표기가 생기면 근거 없는 답입니다. 아래 셀은 그 조건을 검사하고 예상한 실패를 포착합니다.</p>

In [5]:
try:
    assert '[문서 p.' in changed.text
except AssertionError:
    print('예상한 실패: 검색 문맥과 출처가 없습니다.')


예상한 실패: 검색 문맥과 출처가 없습니다.


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [6]:
assert reply.text and changed.text
assert '[문서 p.' not in changed.text
print('검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.')


검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.


<h3>정리</h3>
<p>생성 모델 호출만으로는 사내 문서의 답과 출처를 보장할 수 없습니다. 다음 절부터 문서를 나누고 검색 가능한 형태로 준비합니다.</p>